# Joined (UniProt + ClinVar) EDA

Outer join of the two cleaned RAG parquets on **UniProt primary gene** (first token of `Gene Names`) = **ClinVar gene**.

ClinVar `GeneSymbol` is a `;`-delimited gene list (e.g. `KLLN;LOC130004273;MLDHR;PTEN`), so each variant is **exploded to one row per gene** before joining; the original list is kept in `GeneSymbolFull`.

Built by `python src/build_joined_dataset.py` → `data/processed/joined.parquet`.

Row grain: one row per (protein, variant-gene) pair for shared genes, plus unmatched proteins (ClinVar columns null) and unmatched variant-genes (UniProt columns null). `match_type` flags each row as `both` / `uniprot_only` / `clinvar_only`.

In [1]:
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:

load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
JOINED_PARQUET = project_root / "data/processed/joined.parquet"

In [3]:
df = pd.read_parquet(JOINED_PARQUET)
df.shape

(16794, 37)

In [4]:
df.head(2)

,join_id,Entry,Entry Name,Gene Names,Protein names,Length,Function [CC],Involvement in disease,Subcellular location [CC],Interacts with,...,Assembly,Chromosome,Start,Stop,ReviewStatus,NumberSubmitters,VariationID,GeneSymbolFull,match_type,gene
0,0,A8K2U0,A2ML1_HUMAN,A2ML1 CPAMD9,Alpha-2-macroglobulin-like protein 1 (C3 and P...,1454.0,Is able to inhibit all four classes of protein...,Otitis media (OM) [MIM:166760]: An inflammatio...,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...,None,...,None,None,<NA>,<NA>,None,<NA>,NaN,<NA>,uniprot_only,A2ML1
1,1,Q9NRG9,AAAS_HUMAN,AAAS ADRACALA GL003,Aladin (Adracalin),546.0,Plays a role in the normal development of the ...,Achalasia-addisonianism-alacrima syndrome (AAA...,"SUBCELLULAR LOCATION: Nucleus, nuclear pore co...",None,...,None,None,<NA>,<NA>,None,<NA>,NaN,<NA>,uniprot_only,AAAS


## Join impact: where do the rows come from?

`both` = (protein, variant) pairs on shared genes · `uniprot_only` = proteins with no expert-reviewed variant · `clinvar_only` = variants whose gene has no disease-annotated reviewed protein.

In [5]:
df["match_type"].value_counts(dropna=False)

match_type
both            11551
uniprot_only     5154
clinvar_only       89
Name: count, dtype: int64

In [6]:
# Input rows vs join output, and how many source records actually matched
n_uniprot = df["Entry"].notna().sum()
n_clinvar = df["VariationID"].notna().sum()
uniprot_genes = set(df.loc[df["Entry"].notna(), "gene"].dropna())
clinvar_genes = set(df.loc[df["VariationID"].notna(), "gene"].dropna())

pd.DataFrame({
    "metric": [
        "joined rows (total)",
        "rows carrying a UniProt protein",
        "rows carrying a ClinVar variant",
        "unique genes (any source)",
        "genes in BOTH sources",
        "genes UniProt-only",
        "genes ClinVar-only",
        "distinct proteins kept (Entry)",
        "distinct variants kept (VariationID)",
    ],
    "value": [
        len(df),
        int(n_uniprot),
        int(n_clinvar),
        df["gene"].nunique(),
        len(uniprot_genes & clinvar_genes),
        len(uniprot_genes - clinvar_genes),
        len(clinvar_genes - uniprot_genes),
        df["Entry"].nunique(),
        df["VariationID"].nunique(),
    ],
})

,metric,value
0,joined rows (total),16794
1,rows carrying a UniProt protein,16705
2,rows carrying a ClinVar variant,11640
3,unique genes (any source),5322
4,genes in BOTH sources,142
5,genes UniProt-only,5149
6,genes ClinVar-only,31
7,distinct proteins kept (Entry),5296
8,distinct variants kept (VariationID),11591


## Fan-out: variants per gene (why `both` is so large)

The outer join multiplies each protein by every variant on its gene, so a few high-variant genes dominate the row count.

In [7]:
# Top genes by number of distinct ClinVar variants (matched genes only)
both = df[df["match_type"] == "both"]
both.groupby("gene")["VariationID"].nunique().sort_values(ascending=False).head(15)

gene
BRCA2     2020
BRCA1     1569
RUNX1      656
PAH        515
MLH1       347
GCK        323
MSH2       296
CFTR       238
LDLR       229
GAA        223
CDH1       207
HNF1A      199
ITGA2B     187
DYSF       176
RPGR       145
Name: VariationID, dtype: int64

In [8]:
# Genes with more than one reviewed protein (these also multiply the rows)
multi_protein = (
    df[df["Entry"].notna()].groupby("gene")["Entry"].nunique()
)
multi_protein[multi_protein > 1].sort_values(ascending=False)

gene
GNAS     4
ERCC6    2
MOCS2    2
Name: Entry, dtype: int64

In [9]:
g = 'MOCS2'
df[df['gene'] == g]



,join_id,Entry,Entry Name,Gene Names,Protein names,Length,Function [CC],Involvement in disease,Subcellular location [CC],Interacts with,...,Assembly,Chromosome,Start,Stop,ReviewStatus,NumberSubmitters,VariationID,GeneSymbolFull,match_type,gene
10765,10765,O96007,MOC2B_HUMAN,MOCS2 MCBPE MOCO1,Molybdopterin synthase catalytic subunit (EC 2...,188.0,Catalytic subunit of the molybdopterin synthas...,Molybdenum cofactor deficiency B (MOCODB) [MIM...,"SUBCELLULAR LOCATION: Cytoplasm, cytosol {ECO:...",O96033,...,None,None,<NA>,<NA>,None,<NA>,NaN,<NA>,uniprot_only,MOCS2
10766,10766,O96033,MOC2A_HUMAN,MOCS2 MOCO1,Molybdopterin synthase sulfur carrier subunit ...,88.0,Acts as a sulfur carrier required for molybdop...,Molybdenum cofactor deficiency B (MOCODB) [MIM...,"SUBCELLULAR LOCATION: Cytoplasm, cytosol {ECO:...",O96007,...,None,None,<NA>,<NA>,None,<NA>,NaN,<NA>,uniprot_only,MOCS2


GNAS is a textbook case of a genuinely complex locus: it produces multiple distinct protein products from the same genomic region through alternative promoters and alternative splicing, and it's an imprinted gene (parent-of-origin-specific expression).

ERCC6 CSB is a historical name for ERCC6, stands for the primary disorder linked to its mutation, Cockayne Syndrome B (CS-B).

MOCS2 MCBPE MOCO1 is also an alias for the MOCS2 MOCO1, stands for Molybdenum Cofactor Biosynthesis Protein E.

## What the join adds vs. drops

`clinvar_only` genes = reviewed variants we could **not** attach to a disease-annotated UniProt protein (lost protein context). `uniprot_only` genes = disease proteins with no expert-reviewed variant in the cleaned ClinVar set.

In [10]:
# ClinVar-only genes (variants with no matched protein) and their variant counts
clinvar_only = df[df["match_type"] == "clinvar_only"]
print("clinvar_only genes:", clinvar_only["gene"].nunique())
clinvar_only.groupby("gene")["VariationID"].nunique().sort_values(ascending=False)

clinvar_only genes: 31


gene
MT-TL1                                                               13
RMRP                                                                 13
HBA2                                                                  9
MT-TS1                                                                8
MT-TK                                                                 5
LOC130004273                                                          4
MT-TN                                                                 3
MT-TL2                                                                3
MT-TS2                                                                2
MT-TW                                                                 2
DVL2                                                                  2
MIR6886                                                               2
MT-TH                                                                 2
MT-TF                                                      

In [11]:
# Clinical significance of the variants that now carry protein context (match_type == both)
both["ClinicalSignificance"].value_counts(dropna=False)

ClinicalSignificance
Pathogenic                      4065
Likely benign                   2712
Benign                          2515
Likely pathogenic               2257
Pathogenic/Likely pathogenic       1
Benign/Likely benign               1
Name: count, dtype: int64

## One joined record: protein context + a clinical variant side by side

In [12]:
cols = [
    "gene", "match_type",
    "Entry", "Entry Name", "Protein names", "Involvement in disease",
    "VariationID", "Name", "ClinicalSignificance", "PhenotypeList",
]
df[df["gene"] == "BRCA1"][cols].head(5)

,gene,match_type,Entry,Entry Name,Protein names,Involvement in disease,VariationID,Name,ClinicalSignificance,PhenotypeList
1073,BRCA1,both,P38398,BRCA1_HUMAN,Breast cancer type 1 susceptibility protein (E...,Breast cancer (BC) [MIM:114480]: A common mali...,17661.0,NM_007294.4(BRCA1):c.181T>G (p.Cys61Gly),Pathogenic,"Breast-ovarian cancer, familial, susceptibilit..."
1074,BRCA1,both,P38398,BRCA1_HUMAN,Breast cancer type 1 susceptibility protein (E...,Breast cancer (BC) [MIM:114480]: A common mali...,17670.0,NM_007294.4(BRCA1):c.3119G>A (p.Ser1040Asn),Benign,"Breast-ovarian cancer, familial, susceptibilit..."
1075,BRCA1,both,P38398,BRCA1_HUMAN,Breast cancer type 1 susceptibility protein (E...,Breast cancer (BC) [MIM:114480]: A common mali...,17671.0,NM_007294.4(BRCA1):c.3607C>T (p.Arg1203Ter),Pathogenic,"Breast-ovarian cancer, familial, susceptibilit..."
1076,BRCA1,both,P38398,BRCA1_HUMAN,Breast cancer type 1 susceptibility protein (E...,Breast cancer (BC) [MIM:114480]: A common mali...,17672.0,NM_007294.4(BRCA1):c.3748G>T (p.Glu1250Ter),Pathogenic,"Breast-ovarian cancer, familial, susceptibilit..."
1077,BRCA1,both,P38398,BRCA1_HUMAN,Breast cancer type 1 susceptibility protein (E...,Breast cancer (BC) [MIM:114480]: A common mali...,17675.0,NM_007294.4(BRCA1):c.4327C>T (p.Arg1443Ter),Pathogenic,"Breast-ovarian cancer, familial, susceptibilit..."


What's still unmatched (the remaining 31 clinvar_only genes)
These are genuinely outside the disease-filtered UniProt set, so they should stay unmatched:

- Mitochondrial tRNA genes: MT-TL1 (13), MT-TS1 (8), MT-TK (5), MT-TN, MT-TL2, … (not protein-coding)
- Non-coding RNAs: RMRP (13), MIR6886 (2)
- Uncharacterized loci / others: LOC130004273, LOC106804612
- Others: MLDHR, MKRN2, DVL2, HBA1/HBA2

In [13]:
g = 'MLDHR'
df[df['gene'] == g].iloc[:,19:].values[0]

array(['single nucleotide variant', 'NM_000314.6(PTEN):c.-903G>A',
       'MLDHR', '-', 'Benign', 'Sep 14, 2016',
       'MedGen:CN169374|MedGen:C3661900|MONDO:MONDO:0017623,MeSH:D006223,MedGen:C1959582,Orphanet:306498|MONDO:MONDO:0015356,MeSH:D009386,MedGen:C0027672,Orphanet:140162',
       'PTEN hamartoma tumor syndrome. Hereditary cancer-predisposing syndrome',
       'GRCh38', '10', 87863566, 87863566, 'reviewed by expert panel', 7,
       138837.0, 'KLLN;LOC130004273;MLDHR;PTEN', 'clinvar_only', 'MLDHR'],
      dtype=object)

## Scope characterization (combined)

Combined scope of the linked dataset. The join lets a query match on **either** protein text or variant text and surface a gene's full picture; `both`-rows carry protein context (function, disease) alongside clinical variants.

The probe below shows free-text reachability across both sources at once — a concept is in scope wherever it appears.

In [14]:
print("joined rows:", len(df), "| match_type:", df["match_type"].value_counts().to_dict())
print("genes (any source):", df["gene"].nunique())
print("genes with protein + variant (both):", df.loc[df["match_type"] == "both", "gene"].nunique())

# Combined free-text reachability across protein + variant fields
text_cols = [
    c for c in ["Function [CC]", "Involvement in disease", "Tissue specificity", "Name", "PhenotypeList"]
    if c in df.columns
]
def joined_mentions(keyword):
    text = df[text_cols].astype(str).agg(" ".join, axis=1)
    return int(text.str.contains(keyword, case=False, na=False).sum())

for kw in ["hemoglobin", "cancer", "diabetes", "muscle", "mitochond", "retina"]:
    print(f"{kw:12s} {joined_mentions(kw):6d} joined row(s)")

joined rows: 16794 | match_type: {'both': 11551, 'uniprot_only': 5154, 'clinvar_only': 89}
genes (any source): 5322
genes with protein + variant (both): 142


hemoglobin       52 joined row(s)


cancer         6368 joined row(s)


diabetes        836 joined row(s)


muscle         3618 joined row(s)


mitochond      1343 joined row(s)


retina         1274 joined row(s)
